### A/B Testing

In [22]:
#generate a synthetic datset and use for A/B testing analysis
import numpy as np
import pandas as pd

np.random.seed(0)  # For reproducibility
data = pd.DataFrame({
    'UserID': np.arange(1, 1001),
    'Group': np.random.choice(['A', 'B'], size=1000),
    'OpenEmail': np.random.choice([0, 1], size=1000),    # 0=no, 1=yes
    'ClickdLink': np.random.choice([0, 1], size=1000)   # 0=no, 1=yes
})

df=pd.DataFrame(data)
print("sample A/B testing dataset:\n", df.head())

# # A/B testing analysis
# grouped = df.groupby('Group')['OpenEmail'].mean()
# print(grouped)




sample A/B testing dataset:
    UserID Group  OpenEmail  ClickdLink
0       1     A          1           1
1       2     B          0           1
2       3     B          1           0
3       4     A          0           0
4       5     B          0           0


In [23]:
#calcualte the conversion rate for each group
conversion_rate = df.groupby('Group').mean()
print("Conversion Rate by Group:\n", conversion_rate)


Conversion Rate by Group:
            UserID  OpenEmail  ClickdLink
Group                                   
A      514.790323   0.491935    0.528226
B      486.436508   0.537698    0.519841


In [24]:
#perform statistical test (chi-squared test) to determine if the difference in conversion rates is significant
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['Group'], df['OpenEmail'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")


Chi-squared: 1.9167852705940205, p-value: 0.16621148370378575


###### no significant differnce as p value is higher than 0.05

A/B Testing wth market segmentation

In [25]:
np.random.seed(0)  # For reproducibility
data = pd.DataFrame({
    'UserID': np.arange(1, 1001),
    'AgeGroup': np.random.choice (['18-25', '26-35', '36-45', '46-60+'], size=1000),
    'Group': np.random.choice(['A', 'B'], size=1000),
    'OpenEmail': np.random.choice([0, 1], size=1000),    # 0=no, 1=yes
    'ClickedLink': np.random.choice([0, 1], size=1000)   # 0=no, 1=yes
})

df=pd.DataFrame(data)
print("sample segmented A/B testing dataset:\n", df.head())

sample segmented A/B testing dataset:
    UserID AgeGroup Group  OpenEmail  ClickedLink
0       1    18-25     B          1            1
1       2   46-60+     A          1            1
2       3    26-35     B          0            1
3       4    18-25     A          0            0
4       5   46-60+     A          0            0


Analyze A/B test results by segments

In [26]:
#calculate conversion rate by segments
conversion_rates = df.groupby(['AgeGroup', 'Group']).mean()
print("conversion rates by segments:\n", conversion_rates)

conversion rates by segments:
                     UserID  OpenEmail  ClickedLink
AgeGroup Group                                    
18-25    A      474.419847   0.480916     0.519084
         B      524.153226   0.540323     0.572581
26-35    A      509.818966   0.525862     0.431034
         B      519.956204   0.489051     0.510949
36-45    A      539.793388   0.570248     0.487603
         B      523.975000   0.525000     0.525000
46-60+   A      461.589744   0.572650     0.478632
         B      453.619403   0.500000     0.447761


In [27]:
#perform chi-squared test for each age group
from scipy.stats import chi2_contingency

for age_group in df['AgeGroup'].unique():
    subset = df[df['AgeGroup'] == age_group]
    contingency_table = pd.crosstab(subset['Group'], subset['OpenEmail'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"Age Group: {age_group}, Chi-squared: {chi2}, p-value: {p}")


Age Group: 18-25, Chi-squared: 0.6775845045744691, p-value: 0.41041973420343103
Age Group: 46-60+, Chi-squared: 1.0489140610530638, p-value: 0.30575732095206193
Age Group: 26-35, Chi-squared: 0.20917060962276565, p-value: 0.647418249673346
Age Group: 36-45, Chi-squared: 0.3320392694574061, p-value: 0.5644607525051215


###### no significant differnce as p value is higher than 0.05

In [28]:
#conduct A/B test for segmented data
for age_group in df['AgeGroup'].unique():
    subset = df[df['AgeGroup'] == age_group]
    conversion_rate = subset.groupby('Group')['OpenEmail'].mean()
    print(f"Age Group: {age_group}, Conversion Rates:\n{conversion_rate}\n")

Age Group: 18-25, Conversion Rates:
Group
A    0.480916
B    0.540323
Name: OpenEmail, dtype: float64

Age Group: 46-60+, Conversion Rates:
Group
A    0.57265
B    0.50000
Name: OpenEmail, dtype: float64

Age Group: 26-35, Conversion Rates:
Group
A    0.525862
B    0.489051
Name: OpenEmail, dtype: float64

Age Group: 36-45, Conversion Rates:
Group
A    0.570248
B    0.525000
Name: OpenEmail, dtype: float64



Lift

This a mesaure used to evaluate the effectiveness of a mareting campaign 

In [29]:
#create a synthetic data to conduct lift
np.random.seed(0)  # For reproducibility
data = pd.DataFrame({
    'Group': np.random.choice(['A', 'B'], size=1000),
    'Converted': np.random.choice([0, 1], size=1000),    # 0=no, 1=yes
})

df = pd.DataFrame(data)
print("sample dataset for lift analysis:\n", df.head())


sample dataset for lift analysis:
   Group  Converted
0     A          1
1     B          0
2     B          1
3     A          0
4     B          0


In [30]:
conversion_rates = df.groupby('Group')['Converted'].mean()
print(f"Conversion Rates:\n", conversion_rates)

Conversion Rates:
 Group
A    0.491935
B    0.537698
Name: Converted, dtype: float64


In [31]:
#calculate lift
lift = conversion_rates['B'] / conversion_rates['A']
print(f"Lift: {lift:.2f}")

#Use this (ratio) when you want to say "B converts X times as often as A"

Lift: 1.09


In [32]:
control_rate = conversion_rates['A']
treatment_rate = conversion_rates['B']
lift = (treatment_rate - control_rate) / control_rate
print("Lift", lift)

#Use this (relative %) when you want to say "B converts X% better than A" — this is 
# the more common definition in A/B testing

Lift 0.09302628155087166


In [33]:
#Significance testing for lift
#create contingency table
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['Group'], df['Converted'])
print(f'Contingency table: \n', contingency_table)


Contingency table: 
 Converted    0    1
Group              
A          252  244
B          233  271


In [34]:
#perform chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")


Chi-squared: 1.9167852705940205, p-value: 0.16621148370378575


###### There's a 16.6% probability that the 9.3% lift between A and B happened purely by chance